# 🚀 红宝书 本地微调

## 一键运行，在 Colab 免费 GPU 上微调你的红宝书模型

### 准备工作
1. 将 `training_data_qwen.jsonl` 上传到 Colab（左边文件夹图标）
2. 依次运行下面每个单元格
3. 约 20 分钟后得到一个红宝书人格的模型

### 模型选择
- **Qwen2.5-1.5B**：轻量，显存需求 4GB，CPU 也能跑
- **Qwen2.5-7B**：更强，显存需求 8GB（QLoRA），推荐 T4 免费 GPU

In [ ]:
# [1/5] 安装依赖
!pip install -q unsloth transformers datasets accelerate peft bitsandbytes
!pip install -q xformers --index-url https://download.pytorch.org/whl/cu121 2>/dev/null
print('Done installing')

In [ ]:
# [2/5] 加载模型（改 MODEL_NAME 切换模型）
from unsloth import FastLanguageModel
import torch

# 推荐：Qwen2.5-1.5B（轻量快）或 Qwen2.5-7B（效果更好）
# 改成 unsloth/Llama-3.2-3B-Instruct 也行
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# [3/5] 加载训练数据
from datasets import Dataset
import json

# Colab 上把 training_data_qwen.jsonl 放到 /content/ 目录
DATA_FILE = "/content/training_data_qwen.jsonl"

REDBOOK_SYSTEM = "你是红宝书，用矛盾分析法分析算法问题。解题格式：【调查】输入/输出/约束→【主要矛盾】一句话→【解法】→【代码】→【检验】。语录：集中优势兵力→二分/双指针 星星之火→DP 敌进我退→回溯 群众路线→哈希表"

def formatting_func(examples):
    texts = []
    for instruction, output in zip(examples["instruction"], examples["output"]):
        text = f"<|im_start|>system\n{REDBOOK_SYSTEM}<|im_end|>\n<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
        texts.append(text)
    return texts

data = []
with open(DATA_FILE, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

dataset = Dataset.from_list(data)
print(f"Loaded {len(dataset)} training samples")

In [ ]:
# [4/5] 训练
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    formatting_func=formatting_func,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Training...")
trainer.train()
print("Done!")

In [ ]:
# [5/5] 保存模型
import os

# 保存 LoRA 适配器
model.save_pretrained("redbook-lora")
tokenizer.save_pretrained("redbook-lora")

# 合并为完整模型（可选，文件较大）
print("Saving merged model...")
model.save_pretrained_merged("redbook-merged", tokenizer, save_method="merged_16bit")

print(f"Done! LoRA saved to: redbook-lora/")
print(f"Merged model saved to: redbook-merged/")
print("\nDownload both folders from the left sidebar.")
print("\n=== How to use ===")
print("from unsloth import FastLanguageModel")
print("model, tokenizer = FastLanguageModel.from_pretrained('redbook-lora')")
print("FastLanguageModel.for_inference(model)")
print("# Now call the model with redbook style!")

## 测试红宝书模型

微调完成后，运行这个测试用例验证效果：

In [ ]:
# 测试
FastLanguageModel.for_inference(model)

test_questions = [
    "分析一下两数之和",
    "判断链表是否有环，我没思路",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": REDBOOK_SYSTEM},
        {"role": "user", "content": q}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=400, temperature=0.7)
    reply = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    print(f"\n{'='*50}\nQ: {q}\n{'='*50}\nA: {reply}")